# Un agente con llamadas a herramientas

Notebook opcional del **módulo 20** del curso (en la app: `uv run dev` y la ruta `#/modulo/agentes`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

In [ ]:
import json
import re

from transformers import AutoModelForCausalLM, AutoTokenizer

nombre = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(nombre)
modelo = AutoModelForCausalLM.from_pretrained(nombre, torch_dtype="auto")

In [ ]:
def calculadora(expresion: str) -> str:
    """Evalúa una expresión aritmética.

    Args:
        expresion: la expresión, por ejemplo "2 * (3 + 4)".
    """
    if not re.fullmatch(r"[\d\s+\-*/().]+", expresion):
        return "error: expresión no permitida"
    return str(eval(expresion))  # solo dígitos y operadores, validado arriba


HERRAMIENTAS = {"calculadora": calculadora}

In [ ]:
mensajes = [{"role": "user", "content": "¿Cuánto cuestan 1.5 M de tokens a 0.40 $/M más 0.5 M a 1.60 $/M?"}]
for paso in range(4):
    prompt = tok.apply_chat_template(mensajes, tools=[calculadora], add_generation_prompt=True, tokenize=False)
    x = tok(prompt, return_tensors="pt")
    y = modelo.generate(**x, max_new_tokens=256)[0][x["input_ids"].shape[1]:]
    salida = tok.decode(y, skip_special_tokens=True)
    llamadas = re.findall(r"<tool_call>(.*?)</tool_call>", tok.decode(y), re.S)
    if not llamadas:
        print("RESPUESTA:", salida)
        break
    for ll in llamadas:
        ll = json.loads(ll)
        resultado = HERRAMIENTAS[ll["name"]](**ll["arguments"])
        print("herramienta:", ll, "->", resultado)
        mensajes.append({"role": "assistant", "tool_calls": [{"type": "function", "function": ll}]})
        mensajes.append({"role": "tool", "name": ll["name"], "content": resultado})

Con LangGraph, el mismo bucle se expresa como un grafo con un nodo de modelo, un nodo de herramientas y una arista condicional; un supervisor es un nodo más que decide el siguiente agente (ver el módulo 20).